# PyTorch VAE 从零复现：Encoder、重参数化与 ELBO

Variational Autoencoder 不是给普通 AutoEncoder 随便加一点随机噪声。本 Notebook 用基础 PyTorch 模块实现 Encoder、Decoder 和 VariationalAutoencoder.forward，并从公式到工程验证：

- 均值与 log-variance 两个 posterior head；
- 可反向传播的重参数化；
- Gaussian reconstruction 与 KL divergence；
- beta warm-up、validation checkpoint、确定性重建与随机生成；
- posterior collapse 诊断、OOD reconstruction score；
- 生成权限、随机种子、state_dict 与 normalizer 制品绑定。

数据是受控二维混合高斯，漂亮图形只用于验证实现，不能代表真实图像或文本生成能力。

## 1. 运行与随机性合同

默认 CPU、固定全局种子。训练时必须采样 epsilon；评估 reconstruction 时使用 posterior mean，避免一次随机样本让指标抖动。生成接口接受显式 seed 用于回放，但生产不能把可预测 seed 当安全随机数。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from dataclasses import dataclass, field  # 导入本单元所需的依赖。
from hashlib import sha256  # 导入本单元所需的依赖。
from copy import deepcopy  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。

SEED = 3001  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
np.random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.__version__  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "seed": SEED, "device": str(DEVICE)})  # 执行当前语句以推进本节示例。

## 2. 数据、切分与 train-only 标准化

四个二维簇组成 in-distribution 数据。先固定打乱，再切 500 train、150 validation、150 test。均值和标准差只从 train 拟合。

这不是时间任务，因此采用独立样本切分；若样本来自同一用户、原图增强或同一设备，必须按 family/group 切分而不能逐行随机。

In [ ]:
rng = np.random.default_rng(SEED)  # 计算并保存当前步骤的中间状态。
centers = np.array([[-2.5, -2.0], [-2.0, 2.5], [2.5, -2.2], [2.2, 2.4]], dtype=np.float32)  # 计算并保存当前步骤的中间状态。
points, labels = [], []  # 计算并保存当前步骤的中间状态。
for label, center in enumerate(centers):  # 遍历输入元素以累积或检查结果。
    points.append(rng.normal(center, [0.45, 0.6], size=(200, 2)))  # 计算并保存当前步骤的中间状态。
    labels.extend([label] * 200)  # 执行当前语句以推进本节示例。
raw_points = np.concatenate(points).astype(np.float32)  # 计算并保存当前步骤的中间状态。
labels = np.asarray(labels)  # 计算并保存当前步骤的中间状态。
permutation = rng.permutation(len(raw_points))  # 计算并保存当前步骤的中间状态。
raw_points, labels = raw_points[permutation], labels[permutation]  # 计算并保存当前步骤的中间状态。

train_raw, valid_raw, test_raw = raw_points[:500], raw_points[500:650], raw_points[650:]  # 计算并保存当前步骤的中间状态。
train_labels, valid_labels, test_labels = labels[:500], labels[500:650], labels[650:]  # 计算并保存当前步骤的中间状态。
train_mean = train_raw.mean(axis=0)  # 计算并保存当前步骤的中间状态。
train_std = train_raw.std(axis=0)  # 计算并保存当前步骤的中间状态。
train_x = torch.tensor((train_raw - train_mean) / train_std, dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
valid_x = torch.tensor((valid_raw - train_mean) / train_std, dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
test_x = torch.tensor((test_raw - train_mean) / train_std, dtype=torch.float32)  # 计算并保存当前步骤的中间状态。

assert train_x.shape == (500, 2)  # 用受控断言验证关键不变量。
assert valid_x.shape == test_x.shape == (150, 2)  # 用受控断言验证关键不变量。
assert np.all(train_std > 0)  # 用受控断言验证关键不变量。
assert set(train_labels) == set(valid_labels) == set(test_labels) == {0, 1, 2, 3}  # 用受控断言验证关键不变量。
print({"train": len(train_x), "validation": len(valid_x), "test": len(test_x)})  # 执行当前语句以推进本节示例。

## 3. Encoder：输出分布参数而不是一个点

近似 posterior 写作 q(z|x)=N(mu, diag(sigma squared))。网络输出 mu 与 logvar。使用 log-variance 更容易表达任意正方差，但仍要 clamp，避免 exp(logvar) 溢出。

latent_dim 设为 2 只为了可解释；真实模型的维度必须由 validation 与下游需求选择。

In [ ]:
class Encoder(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_dim=2, hidden_dim=32, latent_dim=2):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.input_dim = input_dim  # 计算并保存当前步骤的中间状态。
        self.backbone = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            nn.Linear(input_dim, hidden_dim), nn.SiLU(),  # 执行当前语句以推进本节示例。
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        self.mu_head = nn.Linear(hidden_dim, latent_dim)  # 计算并保存当前步骤的中间状态。
        self.logvar_head = nn.Linear(hidden_dim, latent_dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 2 or x.shape[1] != self.input_dim:  # 按当前条件选择后续控制路径。
            raise ValueError("expected_Bx2")  # 遇到非法合同立即显式失败。
        hidden = self.backbone(x)  # 计算并保存当前步骤的中间状态。
        mu = self.mu_head(hidden)  # 计算并保存当前步骤的中间状态。
        logvar = self.logvar_head(hidden).clamp(-10.0, 8.0)  # 计算并保存当前步骤的中间状态。
        return mu, logvar  # 返回当前分支计算出的结果。

encoder_probe = Encoder()  # 计算并保存当前步骤的中间状态。
mu_probe, logvar_probe = encoder_probe(torch.randn(7, 2))  # 计算并保存当前步骤的中间状态。
assert mu_probe.shape == logvar_probe.shape == (7, 2)  # 用受控断言验证关键不变量。
assert torch.isfinite(mu_probe).all() and torch.isfinite(logvar_probe).all()  # 用受控断言验证关键不变量。
assert float(logvar_probe.max()) <= 8.0 and float(logvar_probe.min()) >= -10.0  # 用受控断言验证关键不变量。

## 4. Decoder：定义 p(x|z) 的均值

二维连续数据使用固定方差 Gaussian likelihood，decoder 输出 reconstruction mean，因此最后一层不加 sigmoid。若输入是 0 到 1 的二值像素，可改用 Bernoulli logits 与 BCEWithLogits；不能混用输出范围和 likelihood。

In [ ]:
class Decoder(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, latent_dim=2, hidden_dim=32, output_dim=2):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.latent_dim = latent_dim  # 计算并保存当前步骤的中间状态。
        self.output_dim = output_dim  # 计算并保存当前步骤的中间状态。
        self.network = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            nn.Linear(latent_dim, hidden_dim), nn.SiLU(),  # 执行当前语句以推进本节示例。
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),  # 执行当前语句以推进本节示例。
            nn.Linear(hidden_dim, output_dim),  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。

    def forward(self, z):  # 定义本节可复用的核心函数。
        if z.ndim != 2 or z.shape[1] != self.latent_dim:  # 按当前条件选择后续控制路径。
            raise ValueError("expected_BxLatent")  # 遇到非法合同立即显式失败。
        return self.network(z)  # 返回当前分支计算出的结果。

decoder_probe = Decoder()  # 计算并保存当前步骤的中间状态。
assert decoder_probe(torch.randn(9, 2)).shape == (9, 2)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    decoder_probe(torch.randn(2, 3))  # 执行当前语句以推进本节示例。
    raise AssertionError("wrong latent dim must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

## 5. 重参数化与 VariationalAutoencoder.forward

直接从 N(mu,sigma squared) 采样会把随机节点放在梯度路径上。重参数化改写为：

z = mu + exp(0.5 logvar) times epsilon，epsilon 服从标准正态。

随机性移到 epsilon，梯度仍能流向 mu 与 logvar。forward 默认训练时采样、eval 时使用 mu，也允许显式覆盖。

In [ ]:
class VariationalAutoencoder(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_dim=2, hidden_dim=32, latent_dim=2):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)  # 计算并保存当前步骤的中间状态。
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)  # 计算并保存当前步骤的中间状态。
        self.input_dim = int(input_dim)  # 计算并保存当前步骤的中间状态。
        self.hidden_dim = int(hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.latent_dim = int(latent_dim)  # 计算并保存当前步骤的中间状态。

    def reparameterize(self, mu, logvar, sample=True, generator=None):  # 定义本节可复用的核心函数。
        if not sample:  # 按当前条件选择后续控制路径。
            return mu  # 返回当前分支计算出的结果。
        std = torch.exp(0.5 * logvar)  # 计算并保存当前步骤的中间状态。
        epsilon = torch.randn(  # 计算并保存当前步骤的中间状态。
            mu.shape, dtype=mu.dtype, device=mu.device, generator=generator  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        return mu + std * epsilon  # 返回当前分支计算出的结果。

    def forward(self, x, sample=None, generator=None):  # 定义本节可复用的核心函数。
        mu, logvar = self.encoder(x)  # 计算并保存当前步骤的中间状态。
        should_sample = self.training if sample is None else bool(sample)  # 计算并保存当前步骤的中间状态。
        z = self.reparameterize(mu, logvar, should_sample, generator)  # 计算并保存当前步骤的中间状态。
        reconstruction = self.decoder(z)  # 计算并保存当前步骤的中间状态。
        return reconstruction, mu, logvar, z  # 返回当前分支计算出的结果。

vae = VariationalAutoencoder().to(DEVICE)  # 计算并保存当前步骤的中间状态。
vae.train()  # 执行当前语句以推进本节示例。
reconstruction, mu, logvar, z = vae(train_x[:8])  # 计算并保存当前步骤的中间状态。
assert reconstruction.shape == mu.shape == logvar.shape == z.shape == (8, 2)  # 用受控断言验证关键不变量。
vae.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    deterministic_a = vae(train_x[:8])[0]  # 计算并保存当前步骤的中间状态。
    deterministic_b = vae(train_x[:8])[0]  # 计算并保存当前步骤的中间状态。
assert torch.allclose(deterministic_a, deterministic_b)  # 用受控断言验证关键不变量。

## 6. ELBO：显式 Gaussian 方差、reduction 与重参数梯度 oracle

令 decoder 输出固定方差 Gaussian 的均值。这里把方差明确设为 1，并保留归一化常数：

negative log p(x|z) = 0.5 times (squared error / variance + D log(2 pi variance))。

KL 按 latent dimension 求和，再对 batch 求均值。beta=1 时是 Monte Carlo negative ELBO；beta 不等于 1 时是 beta-VAE 加权目标，不能把二者混称。除了总 loss 能 backward，还直接对 z 关于 mu/logvar 求导并核对解析式，避免 decoder 或 KL 的其他梯度掩盖一条断掉的重参数路径。

In [ ]:
GAUSSIAN_VARIANCE = 1.0  # 计算并保存当前步骤的中间状态。

def vae_loss(x, reconstruction, mu, logvar, beta=1.0, gaussian_variance=GAUSSIAN_VARIANCE):  # 定义本节可复用的核心函数。
    if not (x.ndim == 2 and x.shape == reconstruction.shape and mu.shape == logvar.shape):  # 按当前条件选择后续控制路径。
        raise ValueError("loss_shape_mismatch")  # 遇到非法合同立即显式失败。
    if beta < 0 or gaussian_variance <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("invalid_objective_scale")  # 遇到非法合同立即显式失败。
    squared_error = ((reconstruction - x) ** 2).sum(dim=1)  # 计算并保存当前步骤的中间状态。
    gaussian_constant = x.shape[1] * math.log(2 * math.pi * gaussian_variance)  # 计算并保存当前步骤的中间状态。
    reconstruction_nll_per_sample = 0.5 * (squared_error / gaussian_variance + gaussian_constant)  # 计算并保存当前步骤的中间状态。
    kl_per_sample = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)  # 计算并保存当前步骤的中间状态。
    total = (reconstruction_nll_per_sample + beta * kl_per_sample).mean()  # 计算并保存当前步骤的中间状态。
    return total, reconstruction_nll_per_sample.mean(), kl_per_sample.mean()  # 返回当前分支计算出的结果。

zero_mu = torch.zeros(4, 2)  # 计算并保存当前步骤的中间状态。
zero_logvar = torch.zeros(4, 2)  # 计算并保存当前步骤的中间状态。
zero_terms = vae_loss(torch.zeros(4, 2), torch.zeros(4, 2), zero_mu, zero_logvar)  # 计算并保存当前步骤的中间状态。
expected_gaussian_constant = math.log(2 * math.pi)  # 计算并保存当前步骤的中间状态。
assert math.isclose(float(zero_terms[1]), expected_gaussian_constant, rel_tol=1e-6)  # 用受控断言验证关键不变量。
assert zero_terms[2].item() == 0.0  # 用受控断言验证关键不变量。

# 直接 oracle：同一 epsilon 下既核对数值公式，也分别核对 dz/dmu 与 dz/dlogvar。
oracle_mu = torch.tensor([[0.2, -0.4], [0.7, 0.1]], requires_grad=True)  # 计算并保存当前步骤的中间状态。
oracle_logvar = torch.tensor([[-0.6, 0.8], [0.3, -0.2]], requires_grad=True)  # 计算并保存当前步骤的中间状态。
oracle_generator = torch.Generator(device="cpu").manual_seed(771)  # 计算并保存当前步骤的中间状态。
oracle_z = vae.reparameterize(oracle_mu, oracle_logvar, sample=True, generator=oracle_generator)  # 计算并保存当前步骤的中间状态。
epsilon_generator = torch.Generator(device="cpu").manual_seed(771)  # 计算并保存当前步骤的中间状态。
oracle_epsilon = torch.randn(oracle_mu.shape, generator=epsilon_generator)  # 计算并保存当前步骤的中间状态。
expected_z = oracle_mu + torch.exp(0.5 * oracle_logvar) * oracle_epsilon  # 计算并保存当前步骤的中间状态。
grad_mu, grad_logvar = torch.autograd.grad(oracular_sum := oracle_z.sum(), (oracle_mu, oracle_logvar))  # 计算并保存当前步骤的中间状态。
assert torch.allclose(oracle_z, expected_z)  # 用受控断言验证关键不变量。
assert torch.allclose(grad_mu, torch.ones_like(grad_mu))  # 用受控断言验证关键不变量。
assert torch.allclose(grad_logvar, 0.5 * torch.exp(0.5 * oracle_logvar) * oracle_epsilon)  # 用受控断言验证关键不变量。
assert float(grad_logvar.abs().sum()) > 0  # 用受控断言验证关键不变量。

vae.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
reconstruction, mu, logvar, _ = vae(train_x[:16], sample=True)  # 计算并保存当前步骤的中间状态。
loss_probe, _, _ = vae_loss(train_x[:16], reconstruction, mu, logvar, beta=0.05)  # 计算并保存当前步骤的中间状态。
loss_probe.backward()  # 执行当前语句以推进本节示例。
probe_grad_norm = torch.sqrt(sum(  # 计算并保存当前步骤的中间状态。
    (p.grad.detach() ** 2).sum() for p in vae.parameters() if p.grad is not None  # 执行当前语句以推进本节示例。
))  # 执行当前语句以推进本节示例。
vae.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
assert probe_grad_norm > 0 and torch.isfinite(probe_grad_norm)  # 用受控断言验证关键不变量。

## 7. beta warm-up 与 deterministic posterior-mean validation proxy

训练早期把 beta 从 0 逐步升到 0.05，降低 decoder 在学习 reconstruction 前直接忽略 z 的风险。warm-up 是工程启发式，不是 ELBO 定理。

只在 train 做梯度；每 10 步把 z 固定为 posterior mean，在 validation 计算低方差的 **checkpoint proxy**。它适合本受控示例稳定选 checkpoint，但不是对 q(z|x) 期望的 Monte Carlo ELBO，变量和输出都明确使用 `proxy` 命名。test 不参与选择；下一节另用固定 generator 的多样本 Monte Carlo 目标做审计。

In [ ]:
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
vae = VariationalAutoencoder().to(DEVICE)  # 计算并保存当前步骤的中间状态。
optimizer = torch.optim.Adam(vae.parameters(), lr=0.015)  # 计算并保存当前步骤的中间状态。
history, validation_proxy_history = [], []  # 计算并保存当前步骤的中间状态。
best_validation_proxy, best_state = float("inf"), None  # 计算并保存当前步骤的中间状态。

for step in range(301):  # 遍历输入元素以累积或检查结果。
    vae.train()  # 执行当前语句以推进本节示例。
    optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    reconstruction, mu, logvar, _ = vae(train_x, sample=True)  # 计算并保存当前步骤的中间状态。
    beta = 0.05 * min(1.0, step / 80)  # 计算并保存当前步骤的中间状态。
    loss, recon_term, kl_term = vae_loss(train_x, reconstruction, mu, logvar, beta)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    if step == 0:  # 按当前条件选择后续控制路径。
        first_grad_norm = torch.sqrt(sum(  # 计算并保存当前步骤的中间状态。
            (p.grad.detach() ** 2).sum() for p in vae.parameters() if p.grad is not None  # 执行当前语句以推进本节示例。
        ))  # 执行当前语句以推进本节示例。
    torch.nn.utils.clip_grad_norm_(vae.parameters(), 5.0)  # 执行当前语句以推进本节示例。
    optimizer.step()  # 执行当前语句以推进本节示例。
    history.append((float(loss.detach()), float(recon_term.detach()), float(kl_term.detach()), beta))  # 执行当前语句以推进本节示例。

    if step % 10 == 0:  # 按当前条件选择后续控制路径。
        vae.eval()  # 执行当前语句以推进本节示例。
        with torch.no_grad():  # 在受管理的上下文中执行操作。
            valid_recon, valid_mu, valid_logvar, _ = vae(valid_x, sample=False)  # 计算并保存当前步骤的中间状态。
            validation_proxy = float(  # 计算并保存当前步骤的中间状态。
                vae_loss(valid_x, valid_recon, valid_mu, valid_logvar, beta=0.05)[0]  # 计算并保存当前步骤的中间状态。
            )  # 执行当前语句以推进本节示例。
        validation_proxy_history.append((step, validation_proxy))  # 执行当前语句以推进本节示例。
        if validation_proxy < best_validation_proxy:  # 按当前条件选择后续控制路径。
            best_validation_proxy = validation_proxy  # 计算并保存当前步骤的中间状态。
            best_state = deepcopy(vae.state_dict())  # 计算并保存当前步骤的中间状态。

assert best_state is not None  # 用受控断言验证关键不变量。
vae.load_state_dict(best_state)  # 执行当前语句以推进本节示例。
assert history[-1][0] < history[0][0]  # 用受控断言验证关键不变量。
assert first_grad_norm > 0 and torch.isfinite(first_grad_norm)  # 用受控断言验证关键不变量。
print({"train_beta_objective": [round(history[0][0], 4), round(history[-1][0], 4)],  # 执行当前语句以推进本节示例。
       "best_validation_posterior_mean_proxy": round(best_validation_proxy, 4)})  # 执行当前语句以推进本节示例。

## 8. 确定性重建指标与固定随机流 Monte Carlo ELBO 审计

test reconstruction 使用 z=mu，避免采样噪声，并与只输出 train mean 的 baseline 比较 MSE；这仍只是重建指标。另在 validation 上固定 generator，重复采样 q(z|x)，报告 beta=1 的 Monte Carlo negative ELBO，以及与训练一致 beta=0.05 的 Monte Carlo beta-objective。相同 seed 必须逐位复现。

posterior-mean proxy、Monte Carlo ELBO、样本质量是三个不同问题。本例只说明模型学到受控二维结构；真实生成任务还需 importance-weighted likelihood、域指标、人评、隐私与重复记忆审计。

In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def estimate_mc_negative_elbo(model, x, sample_count, seed, beta=1.0):  # 定义本节可复用的核心函数。
    if sample_count < 2:  # 按当前条件选择后续控制路径。
        raise ValueError("mc_requires_at_least_two_samples")  # 遇到非法合同立即显式失败。
    model.eval()  # 执行当前语句以推进本节示例。
    generator = torch.Generator(device=x.device).manual_seed(int(seed))  # 计算并保存当前步骤的中间状态。
    estimates = []  # 计算并保存当前步骤的中间状态。
    for _ in range(sample_count):  # 遍历输入元素以累积或检查结果。
        reconstruction, mu, logvar, _ = model(x, sample=True, generator=generator)  # 计算并保存当前步骤的中间状态。
        estimates.append(vae_loss(x, reconstruction, mu, logvar, beta=beta)[0])  # 计算并保存当前步骤的中间状态。
    stacked = torch.stack(estimates)  # 计算并保存当前步骤的中间状态。
    return float(stacked.mean()), float(stacked.std(unbiased=True))  # 返回当前分支计算出的结果。

vae.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    valid_proxy_reconstruction, valid_proxy_mu, valid_proxy_logvar, _ = vae(valid_x, sample=False)  # 计算并保存当前步骤的中间状态。
    deterministic_validation_proxy = float(  # 计算并保存当前步骤的中间状态。
        vae_loss(valid_x, valid_proxy_reconstruction, valid_proxy_mu, valid_proxy_logvar, beta=0.05)[0]  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。
    test_reconstruction, test_mu, test_logvar, _ = vae(test_x, sample=False)  # 计算并保存当前步骤的中间状态。
validation_mc_negative_elbo = estimate_mc_negative_elbo(vae, valid_x, 32, seed=8801, beta=1.0)  # 计算并保存当前步骤的中间状态。
validation_mc_beta_objective = estimate_mc_negative_elbo(vae, valid_x, 32, seed=8801, beta=0.05)  # 计算并保存当前步骤的中间状态。
validation_mc_repeat = estimate_mc_negative_elbo(vae, valid_x, 32, seed=8801, beta=1.0)  # 计算并保存当前步骤的中间状态。

test_mse = float(((test_reconstruction - test_x) ** 2).mean())  # 计算并保存当前步骤的中间状态。
mean_baseline = torch.zeros_like(test_x)  # 计算并保存当前步骤的中间状态。
baseline_mse = float(((mean_baseline - test_x) ** 2).mean())  # 计算并保存当前步骤的中间状态。
test_kl_per_dim = (  # 计算并保存当前步骤的中间状态。
    -0.5 * (1 + test_logvar - test_mu.pow(2) - test_logvar.exp())  # 执行当前语句以推进本节示例。
).mean(dim=0)  # 计算并保存当前步骤的中间状态。

print({"validation_posterior_mean_proxy": deterministic_validation_proxy,  # 执行当前语句以推进本节示例。
       "validation_mc_negative_elbo_mean_std": validation_mc_negative_elbo,  # 执行当前语句以推进本节示例。
       "validation_mc_beta_objective_mean_std": validation_mc_beta_objective,  # 执行当前语句以推进本节示例。
       "test_mse": test_mse, "mean_baseline_mse": baseline_mse,  # 执行当前语句以推进本节示例。
       "kl_per_dimension": test_kl_per_dim.tolist()})  # 执行当前语句以推进本节示例。
assert math.isclose(deterministic_validation_proxy, best_validation_proxy, rel_tol=0, abs_tol=1e-6)  # 用受控断言验证关键不变量。
assert validation_mc_negative_elbo == validation_mc_repeat  # 用受控断言验证关键不变量。
assert all(math.isfinite(value) for value in validation_mc_negative_elbo + validation_mc_beta_objective)  # 用受控断言验证关键不变量。
assert not math.isclose(deterministic_validation_proxy, validation_mc_beta_objective[0], abs_tol=1e-4)  # 用受控断言验证关键不变量。
assert math.isfinite(test_mse)  # 用受控断言验证关键不变量。
assert test_mse < baseline_mse * 0.35  # 用受控断言验证关键不变量。
assert torch.isfinite(test_kl_per_dim).all()  # 用受控断言验证关键不变量。
assert float(test_kl_per_dim.min()) > 0.01  # 用受控断言验证关键不变量。

## 9. 生成原语、插值与随机种子

底层 prior sampling 从标准正态取得 z，再经 decoder 得到**标准化空间**的值；相同 seed 应可回放，不同 seed 应产生不同样本。这里故意不在底层函数读取全局 normalizer，也不把它当公开服务。后面的 `generate` 必须从受信 artifact 加载模型与 normalizer、完成校验后才能反变换。

插值只是在 latent 空间线性移动，不能自动保证路径具有业务语义。

In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def sample_prior_standardized(model, count, seed):  # 定义本节可复用的核心函数。
    if not 1 <= count <= 64:  # 按当前条件选择后续控制路径。
        raise ValueError("count_out_of_bounds")  # 遇到非法合同立即显式失败。
    device = next(model.parameters()).device  # 计算并保存当前步骤的中间状态。
    generator = torch.Generator(device=device).manual_seed(int(seed))  # 计算并保存当前步骤的中间状态。
    z = torch.randn((count, model.latent_dim), generator=generator, device=device)  # 计算并保存当前步骤的中间状态。
    standardized = model.decoder(z)  # 计算并保存当前步骤的中间状态。
    return z.cpu(), standardized.cpu().numpy()  # 返回当前分支计算出的结果。

z_a, standardized_a = sample_prior_standardized(vae, 6, seed=99)  # 计算并保存当前步骤的中间状态。
z_b, standardized_b = sample_prior_standardized(vae, 6, seed=99)  # 计算并保存当前步骤的中间状态。
_, standardized_c = sample_prior_standardized(vae, 6, seed=100)  # 计算并保存当前步骤的中间状态。
assert np.allclose(standardized_a, standardized_b)  # 用受控断言验证关键不变量。
assert not np.allclose(standardized_a, standardized_c)  # 用受控断言验证关键不变量。
assert standardized_a.shape == (6, 2)  # 用受控断言验证关键不变量。

with torch.no_grad():  # 在受管理的上下文中执行操作。
    endpoints = test_mu[:2]  # 计算并保存当前步骤的中间状态。
    alpha = torch.linspace(0, 1, 7)[:, None]  # 计算并保存当前步骤的中间状态。
    path = (1 - alpha) * endpoints[0] + alpha * endpoints[1]  # 计算并保存当前步骤的中间状态。
    interpolation = vae.decoder(path)  # 计算并保存当前步骤的中间状态。
assert interpolation.shape == (7, 2)  # 用受控断言验证关键不变量。

## 10. Reconstruction score 不是万能 OOD detector

用 validation in-distribution 的 reconstruction 加 0.05 KL 分数选择 99% quantile 阈值，再在 test 与远离训练簇的受控 OOD 上报告。阈值只看 validation；OOD test 不参与调参。

VAE 可能给某些 OOD 很好 reconstruction，因此真实系统要比较 density ratio、ensemble、embedding detector，并报告 ID false positive。

In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def anomaly_score(model, x):  # 定义本节可复用的核心函数。
    reconstruction, mu, logvar, _ = model(x, sample=False)  # 计算并保存当前步骤的中间状态。
    reconstruction_score = ((reconstruction - x) ** 2).sum(dim=1)  # 计算并保存当前步骤的中间状态。
    kl_score = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)  # 计算并保存当前步骤的中间状态。
    return reconstruction_score + 0.05 * kl_score  # 返回当前分支计算出的结果。

valid_score = anomaly_score(vae, valid_x)  # 计算并保存当前步骤的中间状态。
threshold = float(torch.quantile(valid_score, 0.99))  # 计算并保存当前步骤的中间状态。
test_score = anomaly_score(vae, test_x)  # 计算并保存当前步骤的中间状态。
ood_raw = rng.uniform([6.0, 6.0], [9.0, 9.0], size=(120, 2)).astype(np.float32)  # 计算并保存当前步骤的中间状态。
ood_x = torch.tensor((ood_raw - train_mean) / train_std)  # 计算并保存当前步骤的中间状态。
ood_score = anomaly_score(vae, ood_x)  # 计算并保存当前步骤的中间状态。
id_false_positive = float((test_score > threshold).float().mean())  # 计算并保存当前步骤的中间状态。
ood_recall = float((ood_score > threshold).float().mean())  # 计算并保存当前步骤的中间状态。

print({"threshold": threshold, "test_id_false_positive": id_false_positive,  # 执行当前语句以推进本节示例。
       "controlled_ood_recall": ood_recall})  # 执行当前语句以推进本节示例。
assert threshold > 0  # 用受控断言验证关键不变量。
assert 0.0 <= id_false_positive <= 0.05  # 用受控断言验证关键不变量。
assert ood_recall >= 0.9  # 用受控断言验证关键不变量。

## 11. 只接受受信制品的生成与重建接口

公开接口不接受调用方传入的 `nn.Module`。`generate` 与 `reconstruct` 只接收受信 model version，随后从 registry 取模型和 artifact，并在每次推理前调用下一节定义的 `load_trusted_vae` / `validate_artifact`。生成限制 batch、鉴权、记录版本、bundle hash 与 seed；重建用 artifact 内的 normalizer 做正反变换。客户端不能伪造服务端角色或用随机模型冒充已发布版本。

这里只用进程内不可伪造 marker 演示认证边界，不代表真实 token 验证；输出前仍需域内安全策略。

In [ ]:
_ISSUER = object()  # 计算并保存当前步骤的中间状态。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class AuthContext:  # 定义承载本节状态与行为的数据结构。
    tenant: str  # 执行当前语句以推进本节示例。
    roles: frozenset  # 执行当前语句以推进本节示例。
    subject: str  # 执行当前语句以推进本节示例。
    _marker: object = field(repr=False, compare=False)  # 计算并保存当前步骤的中间状态。

def authenticate_demo(token):  # 定义本节可复用的核心函数。
    if token != "signed-generator":  # 按当前条件选择后续控制路径。
        raise PermissionError("authentication_failed")  # 遇到非法合同立即显式失败。
    return AuthContext("tenant-a", frozenset({"model:generate", "model:reconstruct"}),  # 返回当前分支计算出的结果。
                       "service-demo", _ISSUER)  # 执行当前语句以推进本节示例。

def authorize_model_action(auth, artifact, role):  # 定义本节可复用的核心函数。
    if not isinstance(auth, AuthContext) or auth._marker is not _ISSUER:  # 按当前条件选择后续控制路径。
        raise PermissionError("untrusted_auth")  # 遇到非法合同立即显式失败。
    if auth.tenant != artifact["tenant"]:  # 按当前条件选择后续控制路径。
        raise PermissionError("tenant_mismatch")  # 遇到非法合同立即显式失败。
    if role not in auth.roles:  # 按当前条件选择后续控制路径。
        raise PermissionError("missing_role")  # 遇到非法合同立即显式失败。

def generate(count, seed, auth, model_version="vae-2d-v1"):  # 定义本节可复用的核心函数。
    model, artifact = load_trusted_vae(model_version)  # 计算并保存当前步骤的中间状态。
    authorize_model_action(auth, artifact, "model:generate")  # 执行当前语句以推进本节示例。
    _, standardized = sample_prior_standardized(model, count, seed)  # 计算并保存当前步骤的中间状态。
    normalizer = artifact["normalizer"]  # 计算并保存当前步骤的中间状态。
    mean = np.asarray(normalizer["mean"], dtype=np.float32)  # 计算并保存当前步骤的中间状态。
    std = np.asarray(normalizer["std"], dtype=np.float32)  # 计算并保存当前步骤的中间状态。
    raw = standardized * std + mean  # 计算并保存当前步骤的中间状态。
    return raw, {"model_version": artifact["model_version"], "bundle_sha256": artifact["bundle_sha256"],  # 返回当前分支计算出的结果。
                 "seed": int(seed), "count": int(count)}  # 执行当前语句以推进本节示例。

def reconstruct(raw_inputs, auth, model_version="vae-2d-v1"):  # 定义本节可复用的核心函数。
    model, artifact = load_trusted_vae(model_version)  # 计算并保存当前步骤的中间状态。
    authorize_model_action(auth, artifact, "model:reconstruct")  # 执行当前语句以推进本节示例。
    raw = np.asarray(raw_inputs, dtype=np.float32)  # 计算并保存当前步骤的中间状态。
    if raw.ndim != 2 or raw.shape[1] != artifact["model_config"]["input_dim"]:  # 按当前条件选择后续控制路径。
        raise ValueError("reconstruction_shape_mismatch")  # 遇到非法合同立即显式失败。
    if not np.isfinite(raw).all():  # 按当前条件选择后续控制路径。
        raise ValueError("nonfinite_reconstruction_input")  # 遇到非法合同立即显式失败。
    mean = np.asarray(artifact["normalizer"]["mean"], dtype=np.float32)  # 计算并保存当前步骤的中间状态。
    std = np.asarray(artifact["normalizer"]["std"], dtype=np.float32)  # 计算并保存当前步骤的中间状态。
    standardized = torch.tensor((raw - mean) / std, dtype=torch.float32, device=next(model.parameters()).device)  # 计算并保存当前步骤的中间状态。
    model.eval()  # 执行当前语句以推进本节示例。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        reconstructed_standardized = model(standardized, sample=False)[0].cpu().numpy()  # 计算并保存当前步骤的中间状态。
    reconstructed_raw = reconstructed_standardized * std + mean  # 计算并保存当前步骤的中间状态。
    return reconstructed_raw, {"model_version": artifact["model_version"],  # 返回当前分支计算出的结果。
                               "bundle_sha256": artifact["bundle_sha256"],  # 执行当前语句以推进本节示例。
                               "count": len(raw)}  # 执行当前语句以推进本节示例。

AUTH = authenticate_demo("signed-generator")  # 计算并保存当前步骤的中间状态。

## 12. 受信 registry、可执行校验与生产边界

制品绑定完整模型配置、Gaussian variance、beta、train-only normalizer、训练数据 snapshot、代码版本与 state_dict。注册时先重算所有指纹；推理时从 registry 取模型，再次调用 `validate_artifact`。模型对象不在 registry、权重漂移、数据快照变化、配置或 bundle 被改写都会 fail closed。hash 只验证内容一致性，不是签名；生产应在加载阶段验证签名和对象存储摘要，不必把训练集常驻在线服务。

来源：

- Kingma 与 Welling，Auto-Encoding Variational Bayes：https://arxiv.org/abs/1312.6114
- Higgins 等，beta-VAE：https://openreview.net/forum?id=Sy2fzU9gl
- PyTorch Module 官方文档：https://pytorch.org/docs/stable/generated/torch.nn.Module.html
- PyTorch autograd 官方说明：https://pytorch.org/docs/stable/notes/autograd.html

本例没有卷积 decoder、离散 latent、importance-weighted likelihood、FID、人评、版权与隐私审计。

In [ ]:
def array_sha256(array):  # 定义本节可复用的核心函数。
    value = np.ascontiguousarray(array)  # 计算并保存当前步骤的中间状态。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    digest.update(str(value.dtype).encode())  # 执行当前语句以推进本节示例。
    digest.update(json.dumps(list(value.shape)).encode())  # 执行当前语句以推进本节示例。
    digest.update(value.tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

def state_dict_hash(module):  # 定义本节可复用的核心函数。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    for name, tensor in sorted(module.state_dict().items()):  # 遍历输入元素以累积或检查结果。
        value = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(name.encode())  # 执行当前语句以推进本节示例。
        digest.update(str(value.dtype).encode())  # 执行当前语句以推进本节示例。
        digest.update(json.dumps(list(value.shape)).encode())  # 执行当前语句以推进本节示例。
        digest.update(value.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

def artifact_bundle_sha256(value):  # 定义本节可复用的核心函数。
    payload = {key: item for key, item in value.items() if key != "bundle_sha256"}  # 计算并保存当前步骤的中间状态。
    return sha256(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

def vae_model_config(model):  # 定义本节可复用的核心函数。
    return {"input_dim": model.input_dim, "hidden_dim": model.hidden_dim,  # 返回当前分支计算出的结果。
            "latent_dim": model.latent_dim}  # 执行当前语句以推进本节示例。

def validate_runtime_payload(model, value, training_data):  # 定义本节可复用的核心函数。
    if type(model) is not VariationalAutoencoder:  # 按当前条件选择后续控制路径。
        raise TypeError("artifact_architecture_mismatch")  # 遇到非法合同立即显式失败。
    if artifact_bundle_sha256(value) != value.get("bundle_sha256"):  # 按当前条件选择后续控制路径。
        raise RuntimeError("artifact_bundle_hash_mismatch")  # 遇到非法合同立即显式失败。
    if value.get("architecture") != type(model).__name__ or value.get("model_config") != vae_model_config(model):  # 按当前条件选择后续控制路径。
        raise RuntimeError("artifact_model_config_mismatch")  # 遇到非法合同立即显式失败。
    if value.get("state_dict_sha256") != state_dict_hash(model):  # 按当前条件选择后续控制路径。
        raise RuntimeError("artifact_state_dict_mismatch")  # 遇到非法合同立即显式失败。
    if value.get("data_snapshot_sha256") != array_sha256(training_data):  # 按当前条件选择后续控制路径。
        raise RuntimeError("artifact_data_snapshot_mismatch")  # 遇到非法合同立即显式失败。
    if not np.isfinite(training_data).all():  # 按当前条件选择后续控制路径。
        raise RuntimeError("artifact_nonfinite_training_data")  # 遇到非法合同立即显式失败。
    expected_mean = training_data.mean(axis=0)  # 计算并保存当前步骤的中间状态。
    expected_std = training_data.std(axis=0)  # 计算并保存当前步骤的中间状态。
    normalizer = value.get("normalizer", {})  # 计算并保存当前步骤的中间状态。
    if not (np.allclose(normalizer.get("mean"), expected_mean, rtol=0, atol=1e-7) and  # 按当前条件选择后续控制路径。
            np.allclose(normalizer.get("std"), expected_std, rtol=0, atol=1e-7) and  # 计算并保存当前步骤的中间状态。
            np.all(expected_std > 0)):  # 执行当前语句以推进本节示例。
        raise RuntimeError("artifact_normalizer_mismatch")  # 遇到非法合同立即显式失败。
    expected_likelihood = {"family": "gaussian", "variance": GAUSSIAN_VARIANCE,  # 计算并保存当前步骤的中间状态。
                           "reduction": "sum_dimensions_mean_batch"}  # 执行当前语句以推进本节示例。
    if value.get("likelihood") != expected_likelihood or value.get("beta") != 0.05:  # 按当前条件选择后续控制路径。
        raise RuntimeError("artifact_objective_config_mismatch")  # 遇到非法合同立即显式失败。
    return True  # 返回当前分支计算出的结果。

artifact = {  # 计算并保存当前步骤的中间状态。
    "architecture": "VariationalAutoencoder",  # 执行当前语句以推进本节示例。
    "model_version": "vae-2d-v1",  # 执行当前语句以推进本节示例。
    "code_version": "notebook-30-contract-v2",  # 执行当前语句以推进本节示例。
    "tenant": "tenant-a",  # 执行当前语句以推进本节示例。
    "model_config": vae_model_config(vae),  # 执行当前语句以推进本节示例。
    "likelihood": {"family": "gaussian", "variance": GAUSSIAN_VARIANCE,  # 执行当前语句以推进本节示例。
                   "reduction": "sum_dimensions_mean_batch"},  # 执行当前语句以推进本节示例。
    "beta": 0.05,  # 执行当前语句以推进本节示例。
    "normalizer": {"mean": train_mean.tolist(), "std": train_std.tolist()},  # 执行当前语句以推进本节示例。
    "data_snapshot": "gaussian-mixture-train-v1",  # 执行当前语句以推进本节示例。
    "data_snapshot_sha256": array_sha256(train_raw),  # 执行当前语句以推进本节示例。
    "state_dict_sha256": state_dict_hash(vae),  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
artifact["bundle_sha256"] = artifact_bundle_sha256(artifact)  # 计算并保存当前步骤的中间状态。
_TRUSTED_VAE_REGISTRY = {}  # 计算并保存当前步骤的中间状态。

def register_trusted_artifact(model, value, training_data):  # 定义本节可复用的核心函数。
    validate_runtime_payload(model, value, training_data)  # 执行当前语句以推进本节示例。
    version = value["model_version"]  # 计算并保存当前步骤的中间状态。
    if version in _TRUSTED_VAE_REGISTRY:  # 按当前条件选择后续控制路径。
        raise RuntimeError("model_version_already_registered")  # 遇到非法合同立即显式失败。
    _TRUSTED_VAE_REGISTRY[version] = {"model": model, "artifact": deepcopy(value)}  # 计算并保存当前步骤的中间状态。

def validate_artifact(model, value, training_data):  # 定义本节可复用的核心函数。
    trusted = _TRUSTED_VAE_REGISTRY.get(value.get("model_version"))  # 计算并保存当前步骤的中间状态。
    if trusted is None or trusted["model"] is not model:  # 按当前条件选择后续控制路径。
        raise PermissionError("untrusted_model")  # 遇到非法合同立即显式失败。
    if trusted["artifact"] != value:  # 按当前条件选择后续控制路径。
        raise PermissionError("untrusted_artifact")  # 遇到非法合同立即显式失败。
    return validate_runtime_payload(model, value, training_data)  # 返回当前分支计算出的结果。

def load_trusted_vae(model_version):  # 定义本节可复用的核心函数。
    trusted = _TRUSTED_VAE_REGISTRY.get(model_version)  # 计算并保存当前步骤的中间状态。
    if trusted is None:  # 按当前条件选择后续控制路径。
        raise PermissionError("untrusted_model_version")  # 遇到非法合同立即显式失败。
    model, value = trusted["model"], trusted["artifact"]  # 计算并保存当前步骤的中间状态。
    validate_artifact(model, value, train_raw)  # 执行当前语句以推进本节示例。
    return model, value  # 返回当前分支计算出的结果。

register_trusted_artifact(vae, artifact, train_raw)  # 执行当前语句以推进本节示例。
served_samples, serving_trace = generate(5, 123, AUTH)  # 计算并保存当前步骤的中间状态。
served_samples_repeat, _ = generate(5, 123, AUTH)  # 计算并保存当前步骤的中间状态。
reconstructed_raw, reconstruction_trace = reconstruct(test_raw[:5], AUTH)  # 计算并保存当前步骤的中间状态。
assert served_samples.shape == (5, 2)  # 用受控断言验证关键不变量。
assert np.allclose(served_samples, served_samples_repeat)  # 用受控断言验证关键不变量。
assert reconstructed_raw.shape == (5, 2) and np.isfinite(reconstructed_raw).all()  # 用受控断言验证关键不变量。
assert serving_trace["count"] == 5  # 用受控断言验证关键不变量。
assert serving_trace["bundle_sha256"] == artifact["bundle_sha256"]  # 用受控断言验证关键不变量。
assert reconstruction_trace["model_version"] == artifact["model_version"]  # 用受控断言验证关键不变量。
assert validate_artifact(vae, _TRUSTED_VAE_REGISTRY["vae-2d-v1"]["artifact"], train_raw)  # 用受控断言验证关键不变量。

In [ ]:
# 最终回归：目标、随机性、OOD、权限与可信制品均有失败反例
assert train_x.shape[1] == artifact["model_config"]["input_dim"]  # 用受控断言验证关键不变量。
assert vae.encoder.mu_head.out_features == artifact["model_config"]["latent_dim"]  # 用受控断言验证关键不变量。
assert artifact["likelihood"]["variance"] == GAUSSIAN_VARIANCE  # 用受控断言验证关键不变量。
assert history[-1][0] < history[0][0]  # 用受控断言验证关键不变量。
assert first_grad_norm > 0  # 用受控断言验证关键不变量。
assert test_mse < baseline_mse  # 用受控断言验证关键不变量。
assert torch.isfinite(test_mu).all() and torch.isfinite(test_logvar).all()  # 用受控断言验证关键不变量。
assert np.allclose(standardized_a, standardized_b)  # 用受控断言验证关键不变量。
assert ood_recall >= 0.9 and id_false_positive <= 0.05  # 用受控断言验证关键不变量。
assert serving_trace["model_version"] == artifact["model_version"]  # 用受控断言验证关键不变量。
assert reconstruction_trace["bundle_sha256"] == artifact["bundle_sha256"]  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    sample_prior_standardized(vae, 0, 1)  # 执行当前语句以推进本节示例。
    raise AssertionError("zero samples must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。
try:  # 尝试执行可能失败的受控操作。
    vae.encoder(torch.randn(3, 4))  # 执行当前语句以推进本节示例。
    raise AssertionError("wrong input dim must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。
try:  # 尝试执行可能失败的受控操作。
    reconstruct([[float("nan"), 0.0]], AUTH)  # 执行当前语句以推进本节示例。
    raise AssertionError("nonfinite reconstruction input must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。
try:  # 尝试执行可能失败的受控操作。
    generate(1, 7, AuthContext("tenant-b", AUTH.roles, "x", _ISSUER))  # 执行当前语句以推进本节示例。
    raise AssertionError("cross-tenant generation must fail")  # 遇到非法合同立即显式失败。
except PermissionError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "tenant_mismatch"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    generate(1, 7, AuthContext("tenant-a", AUTH.roles, "x", object()))  # 执行当前语句以推进本节示例。
    raise AssertionError("forged auth must fail")  # 遇到非法合同立即显式失败。
except PermissionError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "untrusted_auth"  # 用受控断言验证关键不变量。

trusted_artifact = _TRUSTED_VAE_REGISTRY["vae-2d-v1"]["artifact"]  # 计算并保存当前步骤的中间状态。
parameter = next(vae.parameters())  # 计算并保存当前步骤的中间状态。
parameter_backup = parameter.detach().clone()  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        parameter.add_(0.125)  # 执行当前语句以推进本节示例。
    try:  # 尝试执行可能失败的受控操作。
        generate(1, 7, AUTH)  # 执行当前语句以推进本节示例。
        raise AssertionError("tampered weights must fail closed")  # 遇到非法合同立即显式失败。
    except RuntimeError as error:  # 捕获预期异常并验证失败分支。
        assert str(error) == "artifact_state_dict_mismatch"  # 用受控断言验证关键不变量。
finally:  # 无论结果如何都执行收尾逻辑。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        parameter.copy_(parameter_backup)  # 执行当前语句以推进本节示例。

data_backup = float(train_raw[0, 0])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    train_raw[0, 0] += 0.25  # 计算并保存当前步骤的中间状态。
    try:  # 尝试执行可能失败的受控操作。
        reconstruct(test_raw[:1], AUTH)  # 执行当前语句以推进本节示例。
        raise AssertionError("tampered training snapshot must fail closed")  # 遇到非法合同立即显式失败。
    except RuntimeError as error:  # 捕获预期异常并验证失败分支。
        assert str(error) == "artifact_data_snapshot_mismatch"  # 用受控断言验证关键不变量。
finally:  # 无论结果如何都执行收尾逻辑。
    train_raw[0, 0] = data_backup  # 计算并保存当前步骤的中间状态。

forged_model = VariationalAutoencoder().to(DEVICE)  # 计算并保存当前步骤的中间状态。
forged_model.load_state_dict(vae.state_dict())  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    validate_artifact(forged_model, trusted_artifact, train_raw)  # 执行当前语句以推进本节示例。
    raise AssertionError("unregistered model must fail closed")  # 遇到非法合同立即显式失败。
except PermissionError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "untrusted_model"  # 用受控断言验证关键不变量。

registry_entry = _TRUSTED_VAE_REGISTRY["vae-2d-v1"]  # 计算并保存当前步骤的中间状态。
registered_model = registry_entry["model"]  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    registry_entry["model"] = VariationalAutoencoder().to(DEVICE)  # 计算并保存当前步骤的中间状态。
    try:  # 尝试执行可能失败的受控操作。
        generate(1, 9, AUTH)  # 执行当前语句以推进本节示例。
        raise AssertionError("random registry model replacement must fail closed")  # 遇到非法合同立即显式失败。
    except RuntimeError as error:  # 捕获预期异常并验证失败分支。
        assert str(error) == "artifact_state_dict_mismatch"  # 用受控断言验证关键不变量。
finally:  # 无论结果如何都执行收尾逻辑。
    registry_entry["model"] = registered_model  # 计算并保存当前步骤的中间状态。

tampered_artifact = deepcopy(trusted_artifact)  # 计算并保存当前步骤的中间状态。
tampered_artifact["model_config"]["latent_dim"] = 99  # 计算并保存当前步骤的中间状态。
tampered_artifact["bundle_sha256"] = artifact_bundle_sha256(tampered_artifact)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    validate_artifact(vae, tampered_artifact, train_raw)  # 执行当前语句以推进本节示例。
    raise AssertionError("untrusted artifact must fail closed")  # 遇到非法合同立即显式失败。
except PermissionError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "untrusted_artifact"  # 用受控断言验证关键不变量。

config_probe = VariationalAutoencoder(input_dim=3, hidden_dim=8, latent_dim=4)  # 计算并保存当前步骤的中间状态。
config_reconstruction, config_mu, config_logvar, _ = config_probe(torch.randn(2, 3), sample=False)  # 计算并保存当前步骤的中间状态。
assert config_reconstruction.shape == (2, 3)  # 用受控断言验证关键不变量。
assert config_mu.shape == config_logvar.shape == (2, 4)  # 用受控断言验证关键不变量。
assert validate_artifact(vae, trusted_artifact, train_raw)  # 用受控断言验证关键不变量。
assert generate(1, 11, AUTH)[0].shape == (1, 2)  # 用受控断言验证关键不变量。
print("VAE 目标、重参数梯度、MC ELBO、OOD、可信生成/重建与 fail-closed 制品回归全部通过。")  # 执行当前语句以推进本节示例。